# Observed labels and explicit settlement

## Goal and setup
Create reusable quantities, outcomes and threshold labels from the pinned Premier
League 2024/25 history. Inspect match versus team-match identity and keep
win/loss/push/void/missing settlement separate from numeric encoding.

Use the `misc314_py314` kernel. The [practical guide](../docs/analytics/labels.md)
contains the equations; the [API reference](../docs/analytics/labels_reference.md)
lists all parameters, input schemas and restrictions.

**Assumptions:** these are realized outcomes. No prediction cutoff or warm-up is
applied. W/D/L retains the provider's current-score semantics; settlement rules
are explicitly chosen examples. Dataset assembly and fitting come later.

In [1]:
from pathlib import Path
import pandas as pd
from xdiyo_analytics.data import load_seasons, select_stats
from xdiyo_analytics.histories import build_team_history
from xdiyo_analytics.features import Stat
from xdiyo_analytics.labels import TeamValue, MatchTotal, Outcome, Above, BetOption, create_labels

root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')

In [2]:
data = load_seasons(
    root / 'data/xDiyo_data', ['24_25'], leagues='Premier_League',
    tables=['matches', 'statistics'],
    record_dir=root / 'experiment/initial_population/selections', verify_hashes=True,
)
history = build_team_history(select_stats(data, stats=[
    (period, 'Match overview', 'cornerKicks') for period in ['ALL', '1ST', '2ND']
]))
corners = Stat('ALL', 'Match overview', 'cornerKicks')
print(f'{len(data.matches)} matches; {len(history)} team rows')

380 matches; 760 team rows


## Steps: choose a target and perspective
TeamValue keeps all team rows; MatchTotal and home/away Outcome keep one row per
match in the input's home-row order. Above is strict: \(q>9.5\) gives one, equality
gives zero, and missing stays missing. A dictionary can contain both row units.

In [3]:
labels = create_labels(history, {
    'team_corners': TeamValue(corners),
    'total_corners': MatchTotal(corners),
    'away_outcome': Outcome(perspective='away'),
    'over_9_5': Above(MatchTotal(corners), 9.5),
    'over_10': BetOption(MatchTotal(corners), 'over', line=10),
    'home_win_draw_push': BetOption(Outcome(perspective='home'), 'win', draw='push'),
})
pd.DataFrame([
    {'label': name, 'shape': item.y.shape, 'unit': item.unit, 'perspective': item.perspective}
    for name, item in labels.items()
])

,label,shape,unit,perspective
0,team_corners,"(760, 1)",team_match,team
1,total_corners,"(380, 1)",match,total
2,away_outcome,"(380, 1)",match,away
3,over_9_5,"(380, 1)",match,total
4,over_10,"(380, 1)",match,total
5,home_win_draw_push,"(380, 1)",match,home


## Inspect identity and settlement
Use `identity_columns` for later joins; pandas index labels can repeat. Match
metadata has home/away IDs. An away outcome still follows home-row order.
At an over/under tie the default is a push with missing numeric encoding, while
the settlement table records `push`. Voids require explicit `void_statuses`.
These generic rules do not assert a bookmaker's settlement convention.

In [4]:
target = labels['over_10']
print(target.identity_columns)
print(target.settlement.iloc[:, 0].value_counts().to_dict())
pd.concat([
    target.metadata[['event_id', 'home_id', 'away_id']],
    target.y, target.settlement.add_suffix('_settlement'),
], axis=1).head(6)

('source_league', 'source_season', 'competition_id', 'season_id', 'event_id')
{'win': 181, 'loss': 157, 'push': 42}


,event_id,home_id,away_id,over_10,over_10_settlement
1,12436870,35,43,1.0,win
3,12436871,32,44,1.0,win
5,12436872,42,3,NaN,push
7,12436873,48,30,0.0,loss
9,12436874,39,45,1.0,win
11,12436875,14,60,0.0,loss


In [5]:
periods = create_labels(history, {
    'total_by_period': MatchTotal(Stat(None, 'Match overview', 'cornerKicks')),
})['total_by_period']
assert labels['team_corners'].y.index.equals(history.index)
assert len(labels['total_corners'].y) == len(data.matches) == 380
assert len(labels['team_corners'].y) == 760
assert periods.y.shape == (380, 3)
assert target.y.index.equals(target.metadata.index)
assert target.settlement.shape == target.y.shape
periods.y.head(4)

,total_by_period::team::ALL::Match overview::cornerKicks::value,total_by_period::team::1ST::Match overview::cornerKicks::value,total_by_period::team::2ND::Match overview::cornerKicks::value
1,15.0,3.0,12.0
3,12.0,7.0,5.0
5,10.0,4.0,6.0
7,6.0,2.0,4.0


## Checks and next steps
The observed history contains 380 matches and 760 team rows. All three supplied
periods expand into separate target columns; they are not added together.
The new outputs retain their own unit, perspective and identities.

Choose target columns and an explicit match/team layout before the later
assembler aligns them with features. No model, training split or final X/y
assembly is constructed here. See the
[coverage checklist](../docs/analytics/labels_documentation_checklist.md)
for independent checks, defaults and missing-data behavior.